# RADAR AI - Model Training Pipeline
**Track A: The Edge Vision - FindIT 2026 Phase 2**

Workspace rekayasa dan pelatihan (*training*) ini digunakan untuk mengekstrak fitur kerusakan infrastruktur dari dataset citra satelit/lapangan. Arsitektur yang difokuskan (*MobileNetV3-Small*) dirancang untuk menemukan titik optimal antara ukuran yang ramping (<50 MB) dan kecepatan latensi inferioritas CPU (<3 detik). Langkah ini memungkinkan distribusi ke lapangan tanpa harus bergantung pada API komputasi pihak ketiga (*on-device AI*).

In [1]:
import os 
import sys
import time
import copy
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from collections import Counter

# PyTorch Ecosystem
import torch
import torchvision
import torch.nn as nn 
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

# Utils & Metrics
import timm
from sklearn.metrics import accuracy_score, f1_score

# --- Konfigurasi Awal Sistem ---
# GPU hanya digunakan untuk fase pelatihan. Inference final akan divalidasi dengan CPU-only.
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[*] Training Device: {DEVICE}')

TRAIN_DIR = './train_data'
TEST_DIR = './test_data'
MODEL_DIR = './models'
CLASS_NAMES = ['no-damage', 'minor-damage', 'major-damage', 'destroyed']

# Membuat direktori jika belum ada
for d in [TRAIN_DIR, TEST_DIR, MODEL_DIR]: 
    os.makedirs(d, exist_ok=True)

# Parameter Resolusi & Data Augmentation
IMG_SIZE = 224
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize((256, 256)), 
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(), 
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)), 
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class RADARDataset(Dataset):
    """
    Custom Map-style PyTorch Dataset untuk mengelola Load Data.
    Mendukung pembagian data dinamis (Train/Val/Test) dari format folder standar citra.
    """
    def __init__(self, root, split, transform):
        self.root = Path(root) / split
        self.transform = transform
        self.samples = []
        
        for idx, name in enumerate(CLASS_NAMES):
            class_dir = self.root / name
            if class_dir.exists():
                for img in list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png')):
                    self.samples.append((str(img), idx))
                    
    def __len__(self): 
        return len(self.samples)
        
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform: 
            img = self.transform(img)
        return img, label
        
    def get_weights(self):
        # Mencegah imbalanced class saat sampling data latih
        lbls = [s[1] for s in self.samples]
        cnt = Counter(lbls)
        return torch.DoubleTensor([len(lbls)/(len(cnt)*cnt[l]) for l in lbls])

print('[✓] Setup Konfigurasi & Inisialisasi Class Dataset Berhasil.')

c:\Users\Hype\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[*] Training Device: cpu
[✓] Setup Konfigurasi & Inisialisasi Class Dataset Berhasil.


In [2]:
train_dataset = RADARDataset(TRAIN_DIR, 'train', train_transform)
val_dataset   = RADARDataset(TRAIN_DIR, 'val', val_test_transform)
test_dataset  = RADARDataset(TEST_DIR, 'test', val_test_transform)

if len(train_dataset) > 0:
    sampler = WeightedRandomSampler(train_dataset.get_weights(), len(train_dataset))
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
else:
    train_loader = None
    
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print(f'[✓] Step 2: Global "train_dataset" defined ({len(train_dataset)} images).')

[✓] Step 2: Global "train_dataset" defined (901 images).


In [3]:
import torchvision.models as models
import copy 
import time

# --- Pemuatan Blueprint Jaringan AI (MobileNetV3-Small) ---
# Memilih weights default (pretrained dari ImageNet) untuk mempercepat konvergensi (Transfer Learning)
model = models.mobilenet_v3_small(weights='DEFAULT')

# Modifikasi arsitektur ujung klasifikasi spesifik ke 4 kelas output
# 'no-damage', 'minor-damage', 'major-damage', 'destroyed'
model.classifier[3] = nn.Linear(model.classifier[3].in_features, 4)
model = model.to(DEVICE)

# --- Profiling Baseline CPU Inferencing Time ---
# Simulasi tes waktu untuk constraint < 3 detik 
# Kita hitung rata-rata inferens dari 10 dummy data tensor random berbasis arsitektur ini
cpu_baseline_model = copy.deepcopy(model).cpu().eval()
dummy_sensor_input = torch.randn(1, 3, 224, 224)

# Pemanasan engine (warming up cache)
t0 = time.perf_counter()
for _ in range(10): 
    _ = cpu_baseline_model(dummy_sensor_input)
    
average_inference_time = (time.perf_counter() - t0) / 10

print('-' * 40)
print(f'[✓] Setup MobileNetV3-Small Selesai')
print(f'    Waktu Simulasi CPU Benchmark: {average_inference_time*1000:.1f} ms')
print('-' * 40)

----------------------------------------
[✓] Setup MobileNetV3-Small Selesai
    Waktu Simulasi CPU Benchmark: 12.1 ms
----------------------------------------


In [4]:
# --- Inisiasi Sesi Pelatihan ---
def get_loss_weights(ds):
    # Menyediakan bobot adaptif ke loss function jika kelas sampel sangat tidak seimbang
    cnt = Counter([s[1] for s in ds.samples])
    return torch.FloatTensor([len(ds.samples)/(4*cnt.get(i, 1)) for i in range(4)]).to(DEVICE)

if train_loader is not None:
    # Memanfaatkan Cross Entropy untuk deteksi multi-kelas dan AdamW untuk regularisasi bobot
    criterion = nn.CrossEntropyLoss(weight=get_loss_weights(train_dataset))
    optimizer = optim.AdamW(model.parameters(), lr=3e-4)

    # Durasi Epoch pelatihan (Bisa dinaikkan nanti saat eksekusi GPU untuk akurasi konvergen)
    TOTAL_EPOCHS = 3
    print(f'Memulai Iterasi Epochs Pelatihan: {TOTAL_EPOCHS} epochs')
    print('-' * 40)
    
    for epoch in range(1, TOTAL_EPOCHS + 1):
        model.train()
        total_loss = 0
        total_samples = 0
        
        # Iterasi Batch Data
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            
            # Forward pass & backward propagation
            optimizer.zero_grad()
            outs = model(imgs)
            loss = criterion(outs, lbls)
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item() * imgs.size(0)
            total_samples += imgs.size(0)
            
        epoch_avg_loss = total_loss / total_samples
        print(f'  [ Epoch {epoch}/{TOTAL_EPOCHS} ] - Average Training Loss: {epoch_avg_loss:.4f}')

    # Finalisasi penyimpanan
    model_save_path = './models/model_damage_classifier.pt'
    torch.save(model.state_dict(), model_save_path)
    print('-' * 40)
    print(f'[✓] Pelatihan Selesai. Parameter model dan weight berhasil dibuild/didump pada: {model_save_path}')
else:
    print('[!] Pelatihan dibatalkan: Tidak ditemukan struktur dataset pada folder "train_data/train"')

Memulai Iterasi Epochs Pelatihan: 3 epochs
----------------------------------------
  [ Epoch 1/3 ] - Average Training Loss: 0.3213
  [ Epoch 2/3 ] - Average Training Loss: 0.0736
  [ Epoch 3/3 ] - Average Training Loss: 0.0477
----------------------------------------
[✓] Pelatihan Selesai. Parameter model dan weight berhasil dibuild/didump pada: ./models/model_damage_classifier.pt
